In [1]:
import time
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

/home/russele7/practicum/dle/practicum_dle_sprint_5/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = 'cpu'
model_name = 'bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()
model_cpu = model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
text = 'This is a sample sentence for quantization benchmark.'
inputs = tokenizer(text, return_tensors='pt')

n_runs = 50

with torch.inference_mode():
    # исходный инференс
    start = time.time()
    for _ in range(n_runs):
        _ = model_cpu(**inputs)
    t_orig = (time.time() - start) / n_runs
print(f'FP32 avg inference time (per run): {t_orig:.6f} s')

FP32 avg inference time (per run): 0.048429 s


In [4]:
print('Применяем динамическую квантизацию...')
# Нужно применить quantize_dynamic к model_cpu (квантуем torch.nn.Linear)
quantized_model = torch.quantization.quantize_dynamic(model_cpu, {torch.nn.Linear}, dtype=torch.qint8) # Ваш код здесь

quantized_model.eval()

with torch.inference_mode():
    start = time.time()
    for _ in range(n_runs):
        _ = quantized_model(**inputs)
    t_q = (time.time() - start) / n_runs
print(f'Quantized avg inference time (per run): {t_q:.6f} s')


Применяем динамическую квантизацию...


/tmp/ipykernel_37318/2538344395.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(model_cpu, {torch.nn.Linear}, dtype=torch.qint8) # Ваш код здесь


Quantized avg inference time (per run): 0.012495 s


In [5]:
with torch.inference_mode():
    logits_fp32 = model_cpu(**inputs).logits.detach()
    logits_q = quantized_model(**inputs).logits.detach()

print('Примеры логитов (FP32 vs Quantized):')
print(logits_fp32[0][:6].tolist())
print(logits_q[0][:6].tolist())

Примеры логитов (FP32 vs Quantized):
[0.4955146908760071, 0.08121632039546967]
[0.46280643343925476, 0.05143212899565697]


In [6]:

# Посчитать L2 и max-abs разницу между logits_fp32 и logits_q.
# Формула L2: l2 = ||logits_fp32 - logits_q||_2
# max_abs = max(abs(logits_fp32 - logits_q))
l2 = torch.norm(logits_fp32 - logits_q).item() # Ваш код здесь
max_abs = torch.max(torch.abs(logits_fp32 - logits_q)).item() # Ваш код здесь
print(f'L2 diff: {l2:.6f}, max abs diff: {max_abs:.6f}')

# Сохранение state_dict'ов и сравнение размеров
tmp_fp = 'model_fp32.pth'
tmp_q = 'model_q.pth'
torch.save(model_cpu.state_dict(), tmp_fp)
torch.save(quantized_model.state_dict(), tmp_q)
print('FP32 size (MB):', os.path.getsize(tmp_fp)/1024/1024)
print('Quant size (MB):', os.path.getsize(tmp_q)/1024/1024)

L2 diff: 0.044237, max abs diff: 0.032708
FP32 size (MB): 417.723596572876
Quant size (MB): 173.08575916290283


# QAT

In [7]:
import torch
from transformers import AutoModelForSequenceClassification
from torch.ao.quantization import get_default_qat_qconfig, prepare_qat, convert

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")
model.train()

# Выбираем qconfig, ориентируясь на целевой бэкенд (например, 'fbgemm' для CPU)
qconfig = get_default_qat_qconfig("fbgemm")
model.qconfig = qconfig

# Вставляем fake-quant и observers
prepare_qat(model, inplace=True)

# Fine-tuning: делаем несколько эпох с маленьким lr, следим за валидацией
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
for epoch in range(1, 4):
    for batch in train_loader:
        # inputs - подготовленные батчи
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    # полезно иногда запускать eval на валидации, чтобы наблюдатели собирали статистику

# Перевод в режим eval и конверсия в истинную INT8 модель
model.eval()
model_int8 = convert(model.cpu())
# Далее model_int8 можно экспортировать или запускать в рантайме

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_37318/3177213749.py:13: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more detai

NameError: name 'train_loader' is not defined